# 4.7 前向传播、反向传播和计算图

前向传播按输入到输出的方向计算并保存中间结果；反向传播按相反方向应用链式法则，得到目标函数关于各参数的梯度。

原教材：[4.7 前向传播、反向传播和计算图](https://zh.d2l.ai/chapter_multilayer-perceptrons/backprop.html)

## 学习目标

1. 说清前向传播保存了什么；
2. 使用链式法则推导单隐藏层网络的梯度；
3. 理解动态计算图和梯度累积；
4. 比较手工梯度与 PyTorch 自动微分结果。

## 4.7.1 单隐藏层网络的前向传播

忽略偏置，对单个样本定义：

$$\mathbf z=\mathbf x\mathbf W_1,\quad \mathbf h=\operatorname{ReLU}(\mathbf z),\quad \mathbf o=\mathbf h\mathbf W_2,$$

$$L=\frac12\lVert\mathbf o-\mathbf y\rVert^2,\quad s=\frac{\lambda}{2}(\lVert\mathbf W_1\rVert_F^2+\lVert\mathbf W_2\rVert_F^2),\quad J=L+s.$$

计算图依赖关系是 `x,W1 → z → h`，随后 `h,W2 → o → L`，同时 `W1,W2 → s`，最后 `L,s → J`。

### 中间变量和形状

设输入维数为 $d$、隐藏单元数为 $h$、输出维数为 $q$。采用行向量表示单个样本时：

| 变量 | 形状 | 含义 |
|---|---:|---|
| $\mathbf x$ | $1\times d$ | 输入样本 |
| $\mathbf W_1$ | $d\times h$ | 输入到隐藏层权重 |
| $\mathbf z,\mathbf h$ | $1\times h$ | 预激活与隐藏激活 |
| $\mathbf W_2$ | $h\times q$ | 隐藏层到输出层权重 |
| $\mathbf o,\mathbf y$ | $1\times q$ | 模型输出与标签 |
| $L,s,J$ | 标量 | 数据损失、正则项与总目标 |

小批量输入只需把第一维从 1 换成批量大小 $B$。明确形状不仅能避免矩阵乘法错误，也能帮助判断反向传播中何处需要转置。

In [ ]:
import torch  # 提供张量计算与自动微分功能

torch.manual_seed(42)  # 固定随机种子以复现实验结果
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))  # 自动选择 Colab GPU 或本地 CPU
print('计算设备:', device)  # 输出当前实际使用的计算设备
x = torch.tensor([[1.0, -2.0]], device=device)  # 创建形状为 [1,2] 的单样本输入
y = torch.tensor([[0.5]], device=device)  # 创建形状为 [1,1] 的回归标签
W1 = torch.tensor([[0.3, -0.4], [-0.5, 0.2]], device=device, requires_grad=True)  # 创建需要梯度的第一层权重 [2,2]
W2 = torch.tensor([[0.7], [-0.6]], device=device, requires_grad=True)  # 创建需要梯度的第二层权重 [2,1]
lambd = 0.1  # 设置 L2 正则化强度

In [ ]:
z = x @ W1  # 计算隐藏层预激活值并在计算图中建立矩阵乘法节点
h = torch.relu(z)  # 计算隐藏层激活并保存 ReLU 所需的中间信息
o = h @ W2  # 计算网络的最终输出
data_loss = 0.5 * torch.sum((o - y).pow(2))  # 计算平方误差数据损失
regularization = 0.5 * lambd * (W1.pow(2).sum() + W2.pow(2).sum())  # 计算两层权重的 L2 正则项
objective = data_loss + regularization  # 将数据损失与正则项相加得到目标函数
z.retain_grad()  # 要求 PyTorch 保留非叶子张量 z 的梯度以便观察
h.retain_grad()  # 要求 PyTorch 保留非叶子张量 h 的梯度以便观察
objective.backward()  # 从目标函数出发反向遍历计算图并累积梯度
print('z:', z.detach())  # 输出隐藏层预激活值
print('h:', h.detach())  # 输出 ReLU 处理后的隐藏层激活
print('output:', o.detach())  # 输出网络预测值
print('dJ/dW1:', W1.grad)  # 输出目标函数关于第一层权重的梯度
print('dJ/dW2:', W2.grad)  # 输出目标函数关于第二层权重的梯度

## 4.7.2 手工应用链式法则

从输出向输入依次计算：

$$\frac{\partial L}{\partial\mathbf o}=\mathbf o-\mathbf y,\qquad \frac{\partial J}{\partial\mathbf W_2}=\mathbf h^T\frac{\partial L}{\partial\mathbf o}+\lambda\mathbf W_2,$$

$$\frac{\partial L}{\partial\mathbf h}=\frac{\partial L}{\partial\mathbf o}\mathbf W_2^T,\qquad \frac{\partial L}{\partial\mathbf z}=\frac{\partial L}{\partial\mathbf h}\odot\mathbb 1(\mathbf z>0),$$

$$\frac{\partial J}{\partial\mathbf W_1}=\mathbf x^T\frac{\partial L}{\partial\mathbf z}+\lambda\mathbf W_1.$$

### 反向传播顺序与梯度汇合

反向传播必须从 $J$ 开始，因为某个节点的梯度依赖所有指向目标的下游路径。$\mathbf W_2$ 同时通过输出 $\mathbf o$ 影响数据损失，也直接通过正则项 $s$ 影响目标，因此两条路径的梯度需要相加：

$$\frac{\partial J}{\partial\mathbf W_2}=\underbrace{\mathbf h^T(\mathbf o-\mathbf y)}_{\text{数据拟合路径}}+\underbrace{\lambda\mathbf W_2}_{\text{正则路径}}.$$

$\mathbf W_1$ 同理。计算图中的分叉在反向传播时变成梯度求和，串联则变成链式乘法。这条规则适用于残差连接、共享参数以及任意复杂动态计算图。

ReLU 的局部导数由前向时的 $\mathbf z$ 决定，所以反向传播必须访问前向传播保存的中间量。这正是训练内存随网络深度增长的重要原因。

In [ ]:
dL_do = o.detach() - y  # 根据平方损失公式手工计算损失关于输出的梯度
manual_W2_grad = h.detach().T @ dL_do + lambd * W2.detach()  # 用链式法则计算第二层权重梯度并加入正则项
dL_dh = dL_do @ W2.detach().T  # 将输出梯度反传到隐藏层激活
relu_grad = (z.detach() > 0).to(z.dtype)  # 根据 z 是否大于零构造 ReLU 的局部梯度
dL_dz = dL_dh * relu_grad  # 将上游梯度与 ReLU 局部梯度逐元素相乘
manual_W1_grad = x.T @ dL_dz + lambd * W1.detach()  # 计算第一层权重梯度并加入正则项
print('W1 梯度一致:', torch.allclose(manual_W1_grad, W1.grad))  # 比较手工梯度与自动微分结果
print('W2 梯度一致:', torch.allclose(manual_W2_grad, W2.grad))  # 比较手工梯度与自动微分结果

## 4.7.3 前向传播与反向传播的相互依赖

训练一次小批量通常遵循：清零梯度 → 前向传播 → 计算损失 → 反向传播 → 更新参数。反向传播依赖前向保存的输入和激活，而下一次前向传播又依赖本次更新后的参数，所以训练不能把所有前向传播预先做完再统一反向传播。

反向传播的算术计算量通常与前向传播同阶，但需要保存激活值等中间结果，因此训练比单纯预测占用更多内存。可以通过激活检查点用额外重算换取更低内存。`backward()` 默认把梯度累加到叶子参数的 `.grad`，所以每个训练批次开始前要调用 `optimizer.zero_grad()`。

PyTorch 使用动态计算图：张量运算执行时构建当前图，反向传播后图通常被释放。若控制流依赖输入，不同样本甚至可以生成不同计算图；若确实要对同一图多次反向传播，需要显式保留图，但这会增加内存占用。

## 4.7.4 小结与练习答案

- 前向传播计算并保存中间变量，反向传播按反序应用链式法则；
- 自动微分没有改变微积分规则，只是自动记录依赖并组织梯度计算；
- 正则项会额外贡献 $\lambda\mathbf W$ 梯度；
- 参数梯度默认累积，因此训练循环必须主动清零。

**练习：为什么预测比训练省内存？** 预测使用 `torch.no_grad()`，不必构建反向图或保存求导所需的中间值。